# Part 8 — 5-Fold Group-Aware Cross-Validation

Goal: evaluate the multi-feature regression model across all five GroupKFold splits while keeping every `user_id` entirely inside a single fold.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
ROOT = Path.cwd().parents[1]

data_path = ROOT / "data" / "duolingo_flagship_v5.csv"
split_path = ROOT / "data" / "split_users.csv"

df = pd.read_csv(data_path)
split_df = pd.read_csv(split_path)

cv_users = set(split_df.loc[split_df["split"] == "cv", "user_id"])

df_cv = df[df["user_id"].isin(cv_users)].copy()

print("CV pool shape:", df_cv.shape)
print("CV users:", df_cv["user_id"].nunique())

CV pool shape: (14438, 18)
CV users: 2125


In [3]:
features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups:", groups.nunique())

X shape: (14438, 5)
y shape: (14438,)
groups: 2125


In [4]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx,val_idx) in enumerate(gkf.split(X,y,groups=groups),start=1):
    train_users = set(groups.iloc[train_idx])
    val_users = set(groups.iloc[val_idx])

    overlap = train_users & val_users

    print(
        f"Fold {fold}: "
        f"train rows={len(train_idx)}, "
        f"val rows={len(val_idx)}, "
        f"train users={len(train_users)}, "
        f"val users={len(val_users)}, "
        f"overlap={len(overlap)}"
    )

    assert len(overlap) == 0
    

Fold 1: train rows=11550, val rows=2888, train users=1700, val users=425, overlap=0
Fold 2: train rows=11550, val rows=2888, train users=1700, val users=425, overlap=0
Fold 3: train rows=11550, val rows=2888, train users=1700, val users=425, overlap=0
Fold 4: train rows=11551, val rows=2887, train users=1700, val users=425, overlap=0
Fold 5: train rows=11551, val rows=2887, train users=1700, val users=425, overlap=0


In [5]:
gkf = GroupKFold(n_splits=5)

fold_rmses = []

for fold, (train_idx,val_idx) in enumerate(gkf.split(X,y,groups=groups),start=1):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    pipeline = Pipeline([("scaler",StandardScaler()),("model",LinearRegression())])
    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val,y_pred))
    fold_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")
    

Fold 1 RMSE: 0.284853
Fold 2 RMSE: 0.273630
Fold 3 RMSE: 0.263205
Fold 4 RMSE: 0.262251
Fold 5 RMSE: 0.284558


In [6]:
fold_results = pd.DataFrame({
    "fold": range(1, 6),
    "rmse": fold_rmses
})

fold_results

,fold,rmse
0,1,0.284853
1,2,0.273630
2,3,0.263205
3,4,0.262251
4,5,0.284558


## Conclusion

- `GroupKFold(n_splits=5)` produced five validation folds with completely disjoint user groups.
- Each validation fold contained 425 unseen users.
- User overlap between train and validation was zero in every fold.
- The preprocessing pipeline was fit separately inside each fold.
- Validation RMSE varied across folds:

```text
Fold 1: 0.284853
Fold 2: 0.273630
Fold 3: 0.263205
Fold 4: 0.262251
Fold 5: 0.284558

The variation between folds shows that model performance depends on which unseen users are used for validation.

Therefore, a single train-validation split is not sufficient for a reliable estimate of generalization performance.

In [7]:
cv_mean = np.mean(fold_rmses)
cv_std = np.std(fold_rmses)

print("CV Mean RMSE:", cv_mean)
print("CV Std RMSE:", cv_std)
print(f"5-Fold GroupKFold RMSE: {cv_mean:.6f} ± {cv_std:.6f}")

CV Mean RMSE: 0.27369939258783577
CV Std RMSE: 0.009834044694370473
5-Fold GroupKFold RMSE: 0.273699 ± 0.009834


In [8]:
gkf = GroupKFold(n_splits=5)

baseline_rmses = []

for train_idx, val_idx in gkf.split(X, y, groups=groups):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    baseline_pred = np.full(len(y_val),y_train.mean())
    
    baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))
    baseline_rmses.append(baseline_rmse)

print("Baseline fold RMSEs:", baseline_rmses)
print("Baseline Mean RMSE:", np.mean(baseline_rmses))
print("Baseline Std RMSE:", np.std(baseline_rmses))

Baseline fold RMSEs: [np.float64(0.2874626642713031), np.float64(0.2752383785470428), np.float64(0.26403215411337416), np.float64(0.2641510984719813), np.float64(0.286106882129308)]
Baseline Mean RMSE: 0.27539823550660186
Baseline Std RMSE: 0.010158119962562724


In [9]:
comparison = pd.DataFrame({
    "model": ["Mean Baseline", "Linear Regression"],
    "mean_rmse": [np.mean(baseline_rmses),cv_mean],
    "std_rmse": [np.std(baseline_rmses),cv_std]
})

comparison

,model,mean_rmse,std_rmse
0,Mean Baseline,0.275398,0.010158
1,Linear Regression,0.273699,0.009834


## Conclusion

5-fold group-aware evaluation produced:

- Mean Baseline: `0.275398 ± 0.010158`
- Linear Regression: `0.273699 ± 0.009834`

Both models were evaluated using the same GroupKFold protocol on unseen users.

The linear model slightly outperformed the mean baseline, which indicates that the current features contain some generalizable predictive signal beyond simply predicting the training-set mean.

However, the improvement is small, so this should be treated as an initial signal rather than evidence of a strong model.

Reporting mean ± standard deviation is more informative than reporting a single fold because it summarizes both average performance and variation across different unseen-user groups.

Note: mean ± standard deviation describes fold-to-fold variability; it is not a confidence interval.